In [ ]:
!pip install -q executorch huggingface_hub torch torchao pytorch-tokenizers
!pip list 2>/dev/null | grep -i -E "^(torch|torchao|executorch|pytorch-tokenizers) "

In [ ]:
import torch
print(torch.__version__, torch.__file__)
from torch import Tensor
import executorch.extension.llm.export
print("OK")

In [ ]:
from google.colab import userdata
from huggingface_hub import snapshot_download

snapshot_download(
    "meta-llama/Llama-3.2-1B-Instruct",
    allow_patterns=["original/*"],
    token=userdata.get("HF_TOKEN"),
    local_dir="llama",
)
!ls -la llama/original

In [ ]:
cfg = """
base:
  model_class: llama3_2
  checkpoint: llama/original/consolidated.00.pth
  params: llama/original/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: False
  dtype_override: fp32
export:
  max_seq_length: 2048
  max_context_length: 2048
  output_name: llama3_2_ctx2048.pte
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
backend:
  xnnpack:
    enabled: True
    extended_ops: True
"""
open("export.yaml", "w").write(cfg)
print(cfg)

In [ ]:
!python -m executorch.extension.llm.export.export_llm --config export.yaml
!find / -name "llama3_2_ctx2048.pte" -size +1M 2>/dev/null

In [ ]:
from google.colab import files
files.download("llama3_2_ctx2048.pte")   # if Cell 6's find shows another path, use that path
files.download("llama/original/tokenizer.model")